# 04 · Attention & Transformers from Scratch (Mini-GPT)

| | |
|---|---|
| **Author** | Chandan C |
| **GitHub** | [github.com/chandanc5525](https://github.com/chandanc5525) |
| **Course** | Agentic AI BootCamp · Module 02 – Advanced PyTorch |
| **Repository** | [AgenticAI_BootCamp-/02_advanced_pytorch](https://github.com/chandanc5525/AgenticAI_BootCamp-/tree/main/02_advanced_pytorch) |
| **Level** | Intermediate → Advanced |
| **Prerequisites** | Notebooks 01-03 |

---

### What you will learn
- Scaled dot-product attention, masks, multi-head attention
- Positional encodings (sinusoidal, learned, RoPE)
- Pre-LN Transformer block, residuals, MLP
- Decoder-only GPT with weight tying
- Training a character-level language model
- Decoding: greedy, temperature, top-k, top-p, KV-cache idea
- Parameter counting & scaling intuition

> Every cell runs on a plain CPU. If a GPU (CUDA / Apple MPS) is present, the notebooks use it automatically.

## 0. Setup

In [ ]:
import math, random, time
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(1337); random.seed(1337)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

## 1. Scaled dot-product attention
Each token produces a **query** (what am I looking for?), **key** (what do I contain?) and **value** (what do I give out?).

$$\text{Attention}(Q,K,V)=\text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right)V$$

The $\sqrt{d_k}$ keeps dot-products from growing with dimension and saturating the softmax.

In [ ]:
def attention(q, k, v, mask=None):
    d = q.size(-1)
    scores = q @ k.transpose(-2, -1) / math.sqrt(d)
    if mask is not None:
        scores = scores.masked_fill(~mask, float("-inf"))
    w = scores.softmax(dim=-1)
    return w @ v, w

B, T, d = 1, 6, 16
q = k = v = torch.randn(B, T, d)
causal = torch.tril(torch.ones(T, T, dtype=torch.bool))
out, w = attention(q, k, v, mask=causal)

# PyTorch's fused kernel (FlashAttention / memory-efficient under the hood) must match
ref = F.scaled_dot_product_attention(q, k, v, is_causal=True)
print("matches F.scaled_dot_product_attention:", torch.allclose(out, ref, atol=1e-5))

fig, ax = plt.subplots(1, 2, figsize=(8, 3))
ax[0].imshow(attention(q, k, v)[1][0], cmap="viridis"); ax[0].set_title("Bidirectional (encoder)")
ax[1].imshow(w[0], cmap="viridis"); ax[1].set_title("Causal (decoder / GPT)"); plt.show()

## 2. Multi-head attention
Several attention "heads" operate in parallel in lower-dimensional subspaces so the model can attend to different relations (syntax, coreference, position ...).

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.0, causal=True):
        super().__init__()
        assert d_model % n_heads == 0
        self.h, self.dk, self.causal = n_heads, d_model // n_heads, causal
        self.qkv = nn.Linear(d_model, 3 * d_model, bias=False)   # fused projection
        self.proj = nn.Linear(d_model, d_model, bias=False)
        self.drop = nn.Dropout(dropout); self.p = dropout

    def forward(self, x, pad_mask=None):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)
        q, k, v = (t.view(B, T, self.h, self.dk).transpose(1, 2) for t in (q, k, v))   # (B,H,T,dk)
        attn_mask = None
        if pad_mask is not None:                                   # (B,T) True = real token
            attn_mask = pad_mask[:, None, None, :]                 # (B,1,1,T)
            if self.causal:                                        # combine with causal mask
                attn_mask = attn_mask & torch.tril(torch.ones(T, T, dtype=torch.bool, device=x.device))
        y = F.scaled_dot_product_attention(
            q, k, v, attn_mask=attn_mask,
            dropout_p=self.p if self.training else 0.0,
            is_causal=self.causal and attn_mask is None)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.drop(self.proj(y))

mha = MultiHeadAttention(64, 4)
print(mha(torch.randn(2, 10, 64)).shape)

## 3. Positional information
Attention is permutation-invariant, so position must be injected.

| Method | Used by |
|---|---|
| Sinusoidal (fixed) | Original Transformer |
| Learned absolute | GPT-2, BERT |
| **RoPE** (rotary) | LLaMA, Mistral, Qwen, ... |

In [ ]:
def sinusoidal(T, d):
    pos = torch.arange(T).unsqueeze(1)
    div = torch.exp(torch.arange(0, d, 2) * (-math.log(10000.0) / d))
    pe = torch.zeros(T, d); pe[:, 0::2] = torch.sin(pos * div); pe[:, 1::2] = torch.cos(pos * div)
    return pe

plt.figure(figsize=(6, 3)); plt.imshow(sinusoidal(64, 32).T, aspect="auto", cmap="RdBu"); plt.title("Sinusoidal PE"); plt.xlabel("position"); plt.show()

def rope(x, base=10000.0):
    """Rotary embedding. x: (B, H, T, dk) with even dk."""
    B, H, T, dk = x.shape
    inv = 1.0 / (base ** (torch.arange(0, dk, 2).float() / dk))
    ang = torch.arange(T).float()[:, None] * inv[None]          # (T, dk/2)
    cos, sin = ang.cos(), ang.sin()
    x1, x2 = x[..., 0::2], x[..., 1::2]
    return torch.stack([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1).flatten(-2)

# RoPE property: the dot-product depends only on RELATIVE position
q = torch.randn(1, 1, 1, 16); k = torch.randn(1, 1, 1, 16)
def score(i, j):
    return (rope(q.expand(1, 1, 32, 16))[0, 0, i] @ rope(k.expand(1, 1, 32, 16))[0, 0, j]).item()
print("same offset ->", round(score(3, 1), 4), round(score(10, 8), 4))

## 4. The Transformer block and GPT model
Modern recipe: **Pre-LayerNorm**, residual connections, GELU MLP (4× expansion), weight tying between the token embedding and output head.

In [ ]:
class Block(nn.Module):
    def __init__(self, d, h, p):
        super().__init__()
        self.ln1, self.ln2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.attn = MultiHeadAttention(d, h, p)
        self.mlp = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Linear(4 * d, d), nn.Dropout(p))
    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        return x + self.mlp(self.ln2(x))

class GPT(nn.Module):
    def __init__(self, vocab, block_size=64, d=96, layers=3, heads=4, dropout=0.1):
        super().__init__()
        self.block_size = block_size
        self.tok = nn.Embedding(vocab, d)
        self.pos = nn.Embedding(block_size, d)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList(Block(d, heads, dropout) for _ in range(layers))
        self.ln_f = nn.LayerNorm(d)
        self.head = nn.Linear(d, vocab, bias=False)
        self.head.weight = self.tok.weight                       # weight tying
        self.apply(self._init)
        for n, p in self.named_parameters():                     # GPT-2 scaled residual init
            if n.endswith("proj.weight") or n.endswith("mlp.2.weight"):
                nn.init.normal_(p, 0.0, 0.02 / math.sqrt(2 * layers))

    @staticmethod
    def _init(m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, 0.0, 0.02)
            if isinstance(m, nn.Linear) and m.bias is not None: nn.init.zeros_(m.bias)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = self.drop(self.tok(idx) + self.pos(torch.arange(T, device=idx.device)))
        for blk in self.blocks: x = blk(x)
        logits = self.head(self.ln_f(x))
        loss = None if targets is None else F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

print("params (millions) for tiny config:", sum(p.numel() for p in GPT(65).parameters()) / 1e6)

## 5. Data: a tiny character-level corpus
We repeat a short knowledge text so the model can overfit it in seconds on CPU. To train on bigger data (e.g. Tiny-Shakespeare) just replace `text`.

In [ ]:
text = (
 "PyTorch is a deep learning framework. Tensors hold data and autograd computes gradients. "
 "A transformer uses attention to mix information between tokens. "
 "An agent is a language model that can plan, call tools and use memory to reach a goal. "
 "Retrieval augmented generation fetches documents and gives them to the model as context. "
) * 12

chars = sorted(set(text)); stoi = {c: i for i, c in enumerate(chars)}; itos = {i: c for c, i in stoi.items()}
encode = lambda s: [stoi[c] for c in s]; decode = lambda ids: "".join(itos[i] for i in ids)
data = torch.tensor(encode(text))
n = int(0.9 * len(data)); train_data, val_data = data[:n], data[n:]
print("vocab:", len(chars), "| tokens:", len(data))

BLOCK, BATCH = 64, 32
def get_batch(split):
    d = train_data if split == "train" else val_data
    ix = torch.randint(len(d) - BLOCK - 1, (BATCH,))
    x = torch.stack([d[i:i + BLOCK] for i in ix]); y = torch.stack([d[i + 1:i + BLOCK + 1] for i in ix])
    return x.to(device), y.to(device)
x, y = get_batch("train"); print(x.shape, "| target is x shifted by one:", torch.equal(x[0, 1:], y[0, :-1]))

## 6. Training
AdamW + warm-up/cosine schedule + gradient clipping - the standard LLM recipe, just smaller.

In [ ]:
model = GPT(len(chars), block_size=BLOCK).to(device)
STEPS, LR, WARM = 400, 3e-3, 20
opt = torch.optim.AdamW(model.parameters(), lr=LR, betas=(0.9, 0.95), weight_decay=0.1)
lr_at = lambda s: LR * (s + 1) / WARM if s < WARM else LR * (0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * (s - WARM) / (STEPS - WARM))))

@torch.no_grad()
def estimate(split, iters=10):
    model.eval(); l = [model(*get_batch(split))[1].item() for _ in range(iters)]; model.train(); return sum(l) / len(l)

t0 = time.time(); log = []
for step in range(STEPS):
    for g in opt.param_groups: g["lr"] = lr_at(step)
    xb, yb = get_batch("train")
    _, loss = model(xb, yb)
    opt.zero_grad(set_to_none=True); loss.backward()
    nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
    if step % 100 == 0 or step == STEPS - 1:
        print(f"step {step:4d} | train {estimate('train'):.3f} | val {estimate('val'):.3f} | {time.time()-t0:.0f}s")
print("perplexity ≈", math.exp(estimate("val")))

## 7. Text generation and decoding strategies
The model outputs a distribution over the next token; **how you pick from it** changes behaviour.

| Strategy | Effect |
|---|---|
| Greedy | deterministic, repetitive |
| Temperature `T` | `T<1` sharper, `T>1` more random |
| Top-k | sample only among k most likely tokens |
| Top-p (nucleus) | smallest set whose probability mass ≥ p |

In [ ]:
def sample_next(logits, temperature=1.0, top_k=None, top_p=None):
    if temperature == 0: return logits.argmax(-1, keepdim=True)
    logits = logits / temperature
    if top_k is not None:
        kth = logits.topk(top_k).values[..., -1, None]
        logits = logits.masked_fill(logits < kth, float("-inf"))
    if top_p is not None:
        sl, si = logits.sort(descending=True)
        cum = sl.softmax(-1).cumsum(-1)
        remove = cum - sl.softmax(-1) > top_p                  # keep first token crossing p
        sl = sl.masked_fill(remove, float("-inf"))
        logits = torch.full_like(logits, float("-inf")).scatter(-1, si, sl)
    return torch.multinomial(logits.softmax(-1), 1)

@torch.no_grad()
def generate(model, prompt, max_new=150, **kw):
    model.eval()
    idx = torch.tensor([encode(prompt)], device=device)
    for _ in range(max_new):
        logits, _ = model(idx[:, -model.block_size:])
        idx = torch.cat([idx, sample_next(logits[:, -1], **kw)], dim=1)
    return decode(idx[0].tolist())

torch.manual_seed(0)
print("GREEDY    :", generate(model, "An agent", temperature=0)); print()
print("T=0.8,k=10:", generate(model, "An agent", temperature=0.8, top_k=10)); print()
print("T=1.2,p=.9:", generate(model, "An agent", temperature=1.2, top_p=0.9))

## 8. KV-cache (why inference is fast)
Without a cache every new token recomputes keys/values for the whole prefix – O(T²) per token. With a cache you store `K,V` of past tokens and only compute the new token's `q,k,v`. Production libraries (`transformers`, vLLM) all do this. Minimal demonstration with one attention layer:

In [ ]:
class CachedAttention(nn.Module):
    def __init__(self, d=32, h=4):
        super().__init__(); self.h, self.dk = h, d // h
        self.qkv = nn.Linear(d, 3 * d); self.o = nn.Linear(d, d)
    def forward(self, x, cache=None):
        B, T, C = x.shape
        q, k, v = (t.view(B, T, self.h, self.dk).transpose(1, 2) for t in self.qkv(x).split(C, 2))
        if cache is not None:
            k = torch.cat([cache[0], k], 2); v = torch.cat([cache[1], v], 2)
        causal = cache is None
        y = F.scaled_dot_product_attention(q, k, v, is_causal=causal)
        return self.o(y.transpose(1, 2).reshape(B, T, C)), (k, v)

att = CachedAttention().eval(); x = torch.randn(1, 8, 32)
with torch.no_grad():
    full, _ = att(x)                                            # whole sequence at once
    cache, outs = None, []
    for t in range(8):                                          # token by token with cache
        o, cache = att(x[:, t:t+1], cache); outs.append(o)
print("cached == full:", torch.allclose(full, torch.cat(outs, 1), atol=1e-5))

## 9. Scaling intuition – counting parameters
For a decoder with `L` layers and width `d`: **params ≈ 12 · L · d²** (+ embeddings). GPT-3 (L=96, d=12288) ⇒ ≈ 175 B.

In [ ]:
def approx_params(L, d, vocab=50000): return 12 * L * d * d + vocab * d
for name, L, d in [("our mini", 3, 96), ("GPT-2 small", 12, 768), ("GPT-2 XL", 48, 1600), ("GPT-3", 96, 12288)]:
    p = approx_params(L, d); print(f"{name:12s} ≈ {p/1e9:8.3f} B params | fp16 weights ≈ {p*2/1e9:7.1f} GB")

## Exercises
1. Replace learned positional embeddings with RoPE inside `MultiHeadAttention`.
2. Add a padding mask and train on variable-length sentences.
3. Implement a KV-cache for the full `GPT.generate` and measure the speed-up.
4. Add *repetition penalty* and *beam search* to the decoders.
5. Train on a larger corpus and plot validation loss vs. model size to see scaling behaviour.

---
### Summary & next step
Work through the exercises above before moving on. Next notebook: **05_finetuning_peft_and_efficient_training.ipynb**.

*Notebook by **Chandan C** · [github.com/chandanc5525](https://github.com/chandanc5525) · Agentic AI BootCamp*